In [ ]:
from pathlib import Path
from io import BytesIO
import json, csv, hashlib, math

PACKAGE_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                     if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Open this notebook from the package root or notebooks directory.")

SAVE_PDF = False

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import t
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from IPython.display import display, Image
plt.rcdefaults()

def load_npz(relative_path):
    with np.load(PACKAGE_ROOT / relative_path, allow_pickle=False) as archive:
        return {key: archive[key].copy() for key in archive.files}

def add_panel_labels(figure):
    panels = [axis for axis in figure.axes if axis.get_label() != "<colorbar>"]
    assert len(panels) == 8, "Unexpected panel count"
    for index, axis in enumerate(panels):
        axis.text(-0.16, 1.04, f"({chr(97 + index)})",
                  transform=axis.transAxes, fontsize=10, fontweight="bold",
                  ha="left", va="bottom", clip_on=False, gid="panel_label")
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    records = []
    for axis in panels:
        label = next(text for text in axis.texts if text.get_gid() == "panel_label")
        box = label.get_window_extent(renderer)
        for title in (axis.title, axis._left_title, axis._right_title):
            if title.get_text():
                assert not box.overlaps(title.get_window_extent(renderer)), "Panel label overlaps title"
        records.append({"label": label.get_text(), "axes_position": list(axis.get_position().bounds),
                        "label_bbox_pixels": list(box.bounds)})

def export_figure(figure, figure_id, tight=False):
    add_panel_labels(figure)
    options = {"bbox_inches": "tight"} if tight else {}
    preview = BytesIO()
    figure.savefig(preview, format="png", dpi=180, **options)
    display(Image(data=preview.getvalue()))
    if SAVE_PDF:
        output = PACKAGE_ROOT / "outputs"
        output.mkdir(exist_ok=True)
        figure.savefig(output / f"{figure_id}.pdf",
                       metadata={"CreationDate": None, "ModDate": None}, **options)
        print(f"Saved outputs/{figure_id}.pdf")


In [ ]:
ROOT = PACKAGE_ROOT / "data/Fig3_Fig4"
STYLE={
 "noiseless":("Noiseless","#202020","-"),
 "pauli_y_003":(r"Enc. Pauli-Z $p_z=0.03$","#56B4E9","-"),
 "pauli_y_006":(r"Enc. Pauli-Z $p_z=0.06$","#0072B2","--"),
 "symmetric_0025":(r"Enc. symmetric $p=0.025$","#B26C00","--"),
 "symmetric_0005":(r"Enc. symmetric $p=0.005$","#E69F00","-"),
 "allblock_encoder_0005":(r"Enc. $p=0.005$ + block","#9AA548","-."),
 "thermal_weak":(r"CZ thermal $\tau=0.005$","#B9578E","-."),
 "thermal_matched":(r"CZ thermal $\tau=0.0100503$","#7F2754",(0,(5,2,1,2))),
}
TITLES={
 "noiseless":"Noiseless",
 "pauli_y_003":"Encoding Pauli-Z\n"+r"$p_z=0.03$",
 "pauli_y_006":"Encoding Pauli-Z\n"+r"$p_z=0.06$",
 "symmetric_0025":"Encoding symmetric Pauli\n"+r"$p=0.025$",
 "symmetric_0005":"Encoding symmetric Pauli\n"+r"$p=0.005$",
 "allblock_encoder_0005":"Encoding + block Pauli\n"+r"$p_{\rm enc}=0.005$",
 "thermal_weak":"CZ-only thermal relaxation\n"+r"$\tau=0.005$"+", "+r"$T_2=T_1$",
 "thermal_matched":"CZ-only thermal relaxation\n"+r"$\tau=0.0100503$"+", "+r"$T_2=T_1$",
}
BANDS=[("Low 1-5","#009E73","-"),("Middle 6-15","#0072B2","--"),
       ("High 16-20","#003B5C","-.")]
SUMMARY_BAND_LABELS=[band[0] for band in BANDS]+["High-frequency tail 21-39"]
MANIFEST=json.loads((ROOT/"manifest.json").read_text())
SELECTED_K=MANIFEST["selected_frequencies"]


def style():
    plt.rcParams.update({"font.family":"serif","font.serif":["DejaVu Serif"],
     "font.size":8.5,"axes.titlesize":8.5,"axes.labelsize":8.5,"xtick.labelsize":7.5,
     "ytick.labelsize":7.5,"legend.fontsize":8,"axes.linewidth":0.65,
     "pdf.fonttype":42,"ps.fonttype":42,"savefig.facecolor":"white"})

def load_data():
    result={}
    for c in MANIFEST["conditions"]:
        p=ROOT/c["file"]
        assert hashlib.sha256(p.read_bytes()).hexdigest()==c["sha256"],p
        with np.load(p,allow_pickle=False) as z:
            steps=z["steps"].copy(); seeds=z["seeds"].copy()
            amplitudes=z["amplitude_0_39"].copy()
            ratios=z["band_loss_ratio"].copy()
        np.testing.assert_array_equal(steps,np.arange(1001))
        np.testing.assert_array_equal(seeds,np.arange(42,62))
        assert amplitudes.shape==(20,1001,40) and ratios.shape==(20,1001,4)
        assert np.isfinite(amplitudes).all() and np.isfinite(ratios).all()
        np.testing.assert_allclose(ratios[:,0],1,atol=1e-15)
        result[c["key"]]={"steps":steps,"amplitudes":amplitudes.mean(axis=0),
            "band_ratios":ratios,"band_mean":ratios.mean(axis=0),
            "band_half":t.ppf(.975,19)*ratios.std(axis=0,ddof=1)/np.sqrt(20)}
    assert list(result)==list(STYLE)
    return result

def decorate(ax):
    ax.set_xlim(0,1000)
    ax.tick_params(direction="out",length=2.5,width=.65,pad=2)

def plot_fig3(data):
    style()
    fig,axes=plt.subplots(2,3,figsize=(7.2,5.55),sharex=True,sharey=True)
    for ax,k in zip(axes.flat,SELECTED_K):
        for key,d in data.items():
            label,color,dash=STYLE[key]
            line,=ax.plot(d["steps"],d["amplitudes"][:,k],label=label,color=color,
                         linestyle=dash,lw=.95 if key=="noiseless" else .78,
                         zorder=5 if key=="noiseless" else 3)
            np.testing.assert_array_equal(line.get_ydata(),d["amplitudes"][:,k])
        ax.set_title(r"$k="+str(k)+"$",loc="left",pad=4)
        ax.set_ylim(*MANIFEST["amplitude_ylim"])
        ax.set_xticks([0,200,400,600,800,1000])
        ax.ticklabel_format(axis="y",style="plain",useOffset=False)
        decorate(ax)
    for ax in axes[1]:ax.set_xlabel("Training iteration",labelpad=3)
    for ax in axes[:,0]:ax.set_ylabel("Mean Fourier amplitude",labelpad=3)
    h,l=axes[0,0].get_legend_handles_labels()
    fig.legend(h,l,ncol=2,loc="upper center",bbox_to_anchor=(.52,.995),
               frameon=False,fontsize=8,columnspacing=2.5,handlelength=2.8,handletextpad=.6)
    fig.subplots_adjust(left=.09,right=.98,bottom=.10,top=.78,hspace=.35,wspace=.20)
    return fig

def plot_fig4(data):
    style()
    fig,axes=plt.subplots(2,4,figsize=(8.4,5.3),sharex=True,sharey=True)
    for ax,(key,d) in zip(axes.flat,data.items()):
        for b,(label,color,dash) in enumerate(BANDS):
            mean,half=d["band_mean"][:,b],d["band_half"][:,b]
            line,=ax.plot(d["steps"],mean,color=color,linestyle=dash,lw=1.0,label=label)
            np.testing.assert_array_equal(line.get_ydata(),mean)
            ax.fill_between(d["steps"],np.maximum(mean-half,1e-4),mean+half,
                            color=color,alpha=.13,linewidth=0)
        ax.axhline(1,color=".35",lw=.65,linestyle=(0,(3,2)),zorder=1)
        ax.set_title(TITLES[key],fontsize=8.3,pad=7)
        ax.set_yscale("log");ax.set_ylim(*MANIFEST["band_loss_ylim"])
        ax.set_xticks([0,500,1000]);ax.grid(alpha=.12,which="both");decorate(ax)
    for ax in axes[1]:ax.set_xlabel("Training iteration",labelpad=3)
    for ax in axes[:,0]:ax.set_ylabel(r"Mean relative band loss $R_B(n)$",labelpad=3)
    h,l=axes[0,0].get_legend_handles_labels()
    fig.legend(h,l,loc="upper center",bbox_to_anchor=(.53,.995),ncol=3,
               frameon=False,fontsize=8.5,handlelength=2.8,columnspacing=1.7)
    fig.subplots_adjust(left=.075,right=.98,bottom=.11,top=.86,hspace=.57,wspace=.19)
    return fig

In [ ]:
data = load_data()
for c in MANIFEST["conditions"]:
    raw=load_npz("data/Fig3_Fig4/"+c["file"])
    loss=np.stack([raw["mode_loss_0_39"][:,:,lo:hi+1].sum(-1) for lo,hi in raw["band_limits"]],axis=-1)
    np.testing.assert_allclose(loss/loss[:,:1],raw["band_loss_ratio"],rtol=2e-12,atol=1e-12)

In [ ]:
fig = plot_fig4(data)
export_figure(fig,"Fig4")
plt.close(fig)